In [2]:
import CircuitBuilder
import numpy as np
import pickle
import os
from tqdm import tqdm

In [3]:
# Default 832 code

# 54627381 
def c832(er = 0, nrounds = 1):
    cb = CircuitBuilder.CircuitBuilder(er, er, er, er, er, er)
    # State preparation
    cb.al('R 0 1 2 3 4 5 6 7') # Physical qubits
    cb.al('R 8 9 10 11 12') # Ancilla qubits
    cb.set_ancilla_qubits(qubitslistz = [8, 9, 10, 11], qubitslistx = [12])
    cb.al('H 12')
    # Inject error
    #cb.al('X 2')
    # Warmup round
    cb.time_tick()
    cb.couple_z_check([0, 4, 5, 6], True)
    cb.time_tick()
    cb.couple_z_check([1, 5, 0, 4], True)
    cb.time_tick()
    cb.couple_z_check([2, 6, 1, 0], True)
    cb.time_tick()
    cb.couple_z_check([3, 7, 4, 2], True)
    cb.time_tick()
    cb.couple_x_check([0], True)
    cb.time_tick()
    cb.couple_x_check([1], True)
    cb.time_tick()
    cb.couple_x_check([2], True)
    cb.time_tick()
    cb.couple_x_check([3], True)
    cb.time_tick()
    cb.couple_x_check([4], True)
    cb.time_tick()
    cb.couple_x_check([5], True)
    cb.time_tick()
    cb.couple_x_check([6], True)
    cb.time_tick()
    cb.couple_x_check([7], True)
    cb.al('MR(0) 8 9 10 11')
    cb.al('H 12')
    cb.al('MR(0) 12')
    cb.al('H 12')
    # Noise rounds
    cb.begin_loop(nrounds)
    cb.time_tick()
    cb.couple_z_check([0, 4, 5, 6], True)
    cb.time_tick()
    cb.couple_z_check([1, 5, 0, 4], True)
    cb.time_tick()
    cb.couple_z_check([2, 6, 1, 0], True)
    cb.time_tick()
    cb.couple_z_check([3, 7, 4, 2], True)
    cb.time_tick()
    cb.couple_x_check([0], True)
    cb.time_tick()
    cb.couple_x_check([1], True)
    cb.time_tick()
    cb.couple_x_check([2], True)
    cb.time_tick()
    cb.couple_x_check([3], True)
    cb.time_tick()
    cb.couple_x_check([4], True)
    cb.time_tick()
    cb.couple_x_check([5], True)
    cb.time_tick()
    cb.couple_x_check([6], True)
    cb.time_tick()
    cb.couple_x_check([7], True)
    cb.al('MR(0) 8 9 10 11')
    cb.add_detectors([[-1, -6], [-2, -7], [-3, -8], [-4, -9]])
    cb.al('H 12')
    cb.al('MR(0) 12')
    cb.al('H 12')
    cb.add_detectors([[-1, -6]])
    cb.end_loop()
    # Final measurement of all physical qubits
    cb.al(f'MR({0}) 0 1 2 3 4 5 6 7')

    return cb
# 1234
#     1234
# 23  41
# 3 4 2 1
defaultzschedule = np.asarray([
    [1, 2, 3, 4, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 2, 3, 4],
    [2, 3, 0, 0, 4, 1, 0, 0],
    [3, 0, 4, 0, 2, 0, 1, 0]
])
defaultxschedule = np.asarray([
    [1, 2, 3, 4, 5, 6, 7, 8]
])

def checkvalidschedule(schedule):
    for row in schedule:
        seen = set()
        for val in row:
            if val != 0:
                if val in seen:
                    return False
                seen.add(val)
    for col_idx in range(schedule.shape[1]):
        seen = set()
        for row_idx in range(schedule.shape[0]):
            val = schedule[row_idx][col_idx]
            if val != 0:
                if val in seen:
                    return False
                seen.add(val)
    return True

def randomize_schedule(schedule, max_attempts=1000):
    if max_attempts == 0:
        return None
    for i in range(len(schedule)):
        nonzeros = list(range(1, max(schedule[i]) + 1))
        np.random.shuffle(nonzeros)
        for j in range(len(schedule[i])):
            if schedule[i][j] != 0:
                schedule[i][j] = nonzeros.pop()
    if checkvalidschedule(schedule):
        return schedule, max_attempts
    else:
        return randomize_schedule(schedule, max_attempts - 1)


def windowFiller(schedule, nwindow):
    thiswindow = []
    for row in schedule:
        if nwindow in row:
            thiswindow.append(np.where(row == nwindow)[0][0])
        else:
            thiswindow.append(None)
    return thiswindow

def c832scheduleable(er = 0, nrounds = 1, zschedule = defaultzschedule, xschedule = defaultxschedule):
    zschedule, xschedule = np.asarray(zschedule), np.asarray(xschedule)
    cb = CircuitBuilder.CircuitBuilder(er, er, er, er, er, er)
    cb.al('R 0 1 2 3 4 5 6 7')
    #cb.al('X 0 1') # Flip some physical qubits to start, still must be in codespace
    nancillas = len(zschedule) + len(xschedule)
    ancillastring = 'R ' + ' '.join([f'{i}' for i in range(8, 8 + nancillas)])
    cb.al(ancillastring)
    cb.set_ancilla_qubits(qubitslistz = [8 + i for i in range(len(zschedule))], qubitslistx = [8 + len(zschedule) + i for i in range(len(xschedule))])
    #print(cb.ancilla_qubitsz, cb.ancilla_qubitsx)
    # Warmup round
    for windowz in range(1, np.max(zschedule) + 1):
        #cb.time_tick()
        #print(windowFiller(zschedule, windowz))
        cb.couple_z_check(windowFiller(zschedule, windowz), False)
    cb.al('MR(0) ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsz]))
    cb.al('H ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    for windowx in range(1, np.max(xschedule) + 1):
        #cb.time_tick()
        cb.couple_x_check(windowFiller(xschedule, windowx), False)
    cb.al('H ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.al('MR(0) ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.al('R ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsz]))
    # Noise rounds
    cb.begin_loop(nrounds)
    for windowz in range(1, np.max(zschedule) + 1):
        cb.time_tick()
        cb.couple_z_check(windowFiller(zschedule, windowz), True)
    cb.al('MR(0) ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsz]))
    cb.al('R ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.al('H ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    for windowx in range(1, np.max(xschedule) + 1):
        cb.time_tick()
        cb.couple_x_check(windowFiller(xschedule, windowx), True)
    cb.add_detectors([[-1 - i, -1 - i - nancillas] for i in range(len(cb.ancilla_qubitsz))])
    cb.al('H ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.al('MR(0) ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.add_detectors([[-1 - i, -1 - i - nancillas] for i in range(len(cb.ancilla_qubitsx))])
    cb.al('R ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsz]))
    cb.end_loop()
    # Final measurement of all physical qubits
    cb.al(f'MR({0}) 0 1 2 3 4 5 6 7')
    return cb
    

def detector_builder(mseq, syndromelength = 5, nphysicalqubits = 8, nrounds = 1, mode = 'binary'):
    if len(mseq) != syndromelength * (nrounds + 1) + nphysicalqubits:
        raise ValueError(f"Measurement sequence length must be equal to syndromelength * nrounds + nphysicalqubits")
    syndromes, finalmeasurement = mseq[:-nphysicalqubits], mseq[-nphysicalqubits:]
    if mode == 'string':
        det = []
        for i in range(nrounds):
            det.append(''.join([str(0 if x == y else 1) for x, y in zip(syndromes[i*syndromelength:(i+1)*syndromelength], syndromes[(i+1)*syndromelength:(i+2)*syndromelength])]))
    elif mode == 'binary':
        det = np.zeros(nrounds * syndromelength, dtype = np.bool)
        for i in range(nrounds):
            det[i*syndromelength:(i+1)*syndromelength] = np.array([0 if x == y else 1 for x, y in zip(syndromes[i*syndromelength:(i+1)*syndromelength], syndromes[(i+1)*syndromelength:(i+2)*syndromelength])])
    else:
        raise ValueError(f"Invalid mode: {mode}")

    return det, finalmeasurement
    
def passes_postselection(detector):
    return np.sum(detector) < 1

def toDecimal(binary):
    return int(binary, 2)

def negate(string):
    return ''.join(['1' if x == '0' else '0' for x in string])

In [4]:
nrounds = [1, 2, 4, 6, 8]
ers = [0.0001, 0.000316, 0.001, 0.00316, 0.01]
shots = 1000000
def run_simulation(zschedule = defaultzschedule, xschedule = defaultxschedule, name = 'default'):   
    # Keep track of:
    nshots = np.zeros((len(nrounds), len(ers)), dtype = np.int32)
    detectorfired = np.zeros((len(nrounds), len(ers)), dtype = np.int32)
    readout = np.zeros((len(nrounds), len(ers), 2**8), dtype = np.int32)
    readoutpostselect = np.zeros((len(nrounds), len(ers), 2**8), dtype = np.int32)

    for i in tqdm(range(len(nrounds))):
        for j in tqdm(range(len(ers))):
            cb = c832scheduleable(ers[j], nrounds[i], zschedule=zschedule, xschedule=xschedule)
            sample = cb.sample(nshots = shots)
            for row in sample:
                det, finalmeasurement = detector_builder(row[0], len(defaultzschedule) + 1, 8, nrounds[i], mode = 'binary')
                readoutDecimal = toDecimal(finalmeasurement)
                nshots[i, j] += row[1]
                readout[i, j, readoutDecimal] += row[1]
                if passes_postselection(det):
                    readoutpostselect[i, j, readoutDecimal] += row[1]
                else:
                    detectorfired[i, j] += row[1]
    output = np.zeros((len(nrounds), len(ers), 514), dtype = np.int32)
    output[:, :, 0:256] = readout
    output[:, :, 256:512] = readoutpostselect
    output[:, :, 512] = nshots
    output[:, :, 513] = detectorfired
    output.tofile(f'Data/output_{name}.bin')    


In [5]:
cb = c832scheduleable(0.0001, 1, zschedule=defaultzschedule, xschedule=defaultxschedule)
sample = cb.sample(100000)
for row in sample:
    print(row, detector_builder(row[0], len(defaultzschedule) + 1, 8, 1, mode = 'string')[0])

('000000000000000000', 24631) ['00000']
('000000000000000001', 7) ['00000']
('000000000000000010', 10) ['00000']
('000000000000000011', 4) ['00000']
('000000000000000100', 11) ['00000']
('000000000000000111', 1) ['00000']
('000000000000001000', 12) ['00000']
('000000000000001111', 3) ['00000']
('000000000000010000', 4) ['00000']
('000000000000011111', 2) ['00000']
('000000000000100000', 23) ['00000']
('000000000000111111', 2) ['00000']
('000000000001000000', 8) ['00000']
('000000000001111111', 17) ['00000']
('000000000010000000', 19) ['00000']
('000000000010111111', 15) ['00000']
('000000000011011111', 15) ['00000']
('000000000011101111', 11) ['00000']
('000000000011110000', 1) ['00000']
('000000000011110111', 10) ['00000']
('000000000011111000', 2) ['00000']
('000000000011111011', 11) ['00000']
('000000000011111100', 1) ['00000']
('000000000011111101', 8) ['00000']
('000000000011111110', 5) ['00000']
('000000000011111111', 24510) ['00000']
('000000000100000000', 70) ['00001']
('000000

In [38]:
run_simulation(name = 'defaultnoiselessX')

100%|██████████| 5/5 [01:41<00:00, 20.26s/it]


In [7]:
singlepromotionschedules = pickle.load(open('singlepromotionschedules.pkl', 'rb'))
doublepromotionschedules = pickle.load(open('doublepromotionschedules.pkl', 'rb'))


In [ ]:
rand = np.random.randint(0, len(doublepromotionschedules)) 
zschedule = defaultzschedule
xschedule = doublepromotionschedules[rand]
randzschedule, nz = randomize_schedule(zschedule, 1000)
randxschedule, nx = randomize_schedule(xschedule, 1000)
print(nz, nx)
print(randzschedule)
print(randxschedule)
cb = c832scheduleable(0.001, 1, zschedule=randzschedule, xschedule=randxschedule)
cb.get_circuit().diagram()

998 999
[[2 3 1 4 0 0 0 0]
 [0 0 0 0 2 3 1 4]
 [3 4 0 0 1 2 0 0]
 [1 0 2 0 4 0 3 0]]
[[0 4 0 2 0 1 0 3]
 [1 0 0 4 0 3 2 0]
 [2 0 3 0 4 0 1 0]]


/-----------------------------------------------------------------------------------\ /----------------------------------------------------------------------------------------------------\ /----------------------------------------------------------------------------------------------------\ /----------------------------------------------------------------------------------------------------\ /--------------------------------------------------------------------------------------------------------------------\ /-------------------------------------------------------------------------------\ /-------------------------------------------------------------------------------\ /-------------------------------------------------------------------------------\ /---------------------------------------------------------------------------------------------------------------------------------------------------\
 q0: -R-------@-@-----------@--------------------------X-------X----------------------------DEPOLARIZE1(0.001)-------@----------------------------------------------------------DEPOLARIZE2(0.001)-DEPOLARIZE1(0.001)-@-------DEPOLARIZE2(0.001)----------------------------------------------------------DEPOLARIZE1(0.001)-----@-----------------------------------------DEPOLARIZE2(0.001)--------------------DEPOLARIZE1(0.001)-----------------------------------------------------------------------------------------------------DEPOLARIZE1(0.001)---X----------------------DEPOLARIZE2(0.001)--------------------DEPOLARIZE1(0.001)-----X---------------------------------------DEPOLARIZE2(0.001)-DEPOLARIZE1(0.001)----------------------------------------------------------------DEPOLARIZE1(0.001)---------------------------------------------------------------------------------------------------------MR(0):rec[14]--------------
              | |           |                          |       |                                                     |                                                          |                                     |       |                                                                                                  |                                         |                                                                                                                                                                                 |                      |                                                            |                                       |
 q1: -R-------|-|-------@---|-------@------------------|-------|-------X--------------------DEPOLARIZE1(0.001)-------|----------------------------------------------------------|------------------DEPOLARIZE1(0.001)-|-------|---------------------------------------------------------------------------DEPOLARIZE1(0.001)-@---|---DEPOLARIZE2(0.001)--------------------|-------------------------------------DEPOLARIZE1(0.001)-----@-----------------------------------------DEPOLARIZE2(0.001)------------------------------------DEPOLARIZE1(0.001)---|----------------------|-------------------------------------DEPOLARIZE1(0.001)-----|---------------------------------------|------------------DEPOLARIZE1(0.001)----------------------------------------------------------------DEPOLARIZE1(0.001)-X-----DEPOLARIZE2(0.001)--------------------------------------------------------------------------------MR(0):rec[15]--------------
              | |       |   |       |                  |       |       |                                             |                                                          |                                     |       |                                                                                              |   |   |                                     |                                                            |                                         |                                                                          |                      |                        

In [48]:
sample = cb.sample(1000000)
for row in sample:
    print(row, detector_builder(row[0], 7, 8, 1, mode = 'string')[0])

('0000000000000000000000', 14003) ['0000000']
('0000000000000000000001', 55) ['0000000']
('0000000000000000000010', 37) ['0000000']
('0000000000000000000100', 28) ['0000000']
('0000000000000000000101', 1) ['0000000']
('0000000000000000000110', 6) ['0000000']
('0000000000000000001000', 38) ['0000000']
('0000000000000000001010', 12) ['0000000']
('0000000000000000001100', 1) ['0000000']
('0000000000000000010000', 46) ['0000000']
('0000000000000000010001', 1) ['0000000']
('0000000000000000010010', 1) ['0000000']
('0000000000000000010011', 26) ['0000000']
('0000000000000000010100', 1) ['0000000']
('0000000000000000010110', 28) ['0000000']
('0000000000000000010111', 7) ['0000000']
('0000000000000000011010', 45) ['0000000']
('0000000000000000100000', 29) ['0000000']
('0000000000000000100001', 1) ['0000000']
('0000000000000000100010', 1) ['0000000']
('0000000000000000100011', 32) ['0000000']
('0000000000000000100100', 11) ['0000000']
('0000000000000000100101', 38) ['0000000']
('000000000000000

In [11]:
import pickle

with open('sample.pkl', 'wb') as f:
    pickle.dump(sample, f)

In [5]:
schedulenames = [f for f in os.listdir('schedules') if os.path.isfile(os.path.join('schedules', f))]
print(schedulenames[0][9:17])

10100000


In [13]:
cb.get_circuit().diagram()

/-\ /-----------------------\ /-----------------------\ /-----------------------\ /-------------------------------------------------------------------\ /-----------------------------------------------------------------------------------------------\ /-----------------------------------------------------------------------------------------------\ /-----------------------------------------------------------------------------------------------\ /-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------\
 q0: -R---DEPOLARIZE1(0.01)-@-------DEPOLARIZE1(0.01)-----@---DEPOLARIZE1(0.01)-------@-DEPOLARIZE1(0.01)---------X-------------------------------------------DEPOLARIZE1(0.01)-@-------DEPOLARIZE2(0.01)-------------------------------------------------------DEPOLARIZE1(0.01)-----@---------------------------------------DEPOLARIZE2(0.01)-------------------DEPOLARIZE1(0.01)-------@-------------------------------------------------------DEPOLARIZE2(0.01)-DEPOLARIZE1(0.01)---------------------------------------------------------------------------------X---------------------------------------------------------------------MR(0):rec[10]-------------
                            |                             |                           |                           |                                                             |       |                                                                                             |                                       |                                                           |                                                       |                                                                                                                   |
 q1: -R---DEPOLARIZE1(0.01)-|-------DEPOLARIZE1(0.01)-@---|---DEPOLARIZE1(0.01)-----@-|-DEPOLARIZE1(0.01)---------|-X-----------------------------------------DEPOLARIZE1(0.01)-|-------|-----------------------------------------------------------------------DEPOLARIZE1(0.01)-@---|---DEPOLARIZE2(0.01)-------------------|-----------------------------------DEPOLARIZE1(0.01)-----@-|-------------------------------------DEPOLARIZE2(0.01)-|-----------------DEPOLARIZE1(0.01)---------------------------------------------------------------------------------|-X-------------------------------------------------------------------MR(0):rec[11]-------------
                            |                         |   |                         | |                           | |                                                           |       |                                                                                         |   |   |                                   |                                                         | |                                     |                 |                                                                                                                   | |
 q2: -R---DEPOLARIZE1(0.01)-|-------DEPOLARIZE1(0.01)-|---|---DEPOLARIZE1(0.01)-@---|-|-DEPOLARIZE1(0.01)-------@-|-|-X---------------------------------------DEPOLARIZE1(0.01)-|-------|-----------------------------------------------------------------------DEPOLARIZE1(0.01)-|---|---|-----------------------------------|-----------------------------------DEPOLARIZE1(0.01)-@---|-|-DEPOLARIZE2(0.01)-------------------|-----------------|-----------------DEPOLARIZE1(0.01)-------@-------------------------------------------------------DEPOLARIZE2(0.01)-|-|-X-----------------------------------------------------------------MR(0):rec[12]-------------
                            |                         |   |                     |   | |                         | | | |                                                         |       |                                                                                         |   |   |                              

In [5]:
xschedule = np.asarray([
    [1, 2, 3, 4, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 2, 3, 4]
])
cb = c832scheduleable(0.01, 1, zschedule=defaultzschedule, xschedule=defaultxschedule)
sample = cb.sample(10000000)
for row in sample:
    print(row, detector_builder(row[0], 5, 8, 1, mode = 'string')[0])

('000000000000000000', 962990) ['00000']
('000000000000000001', 2228) ['00000']
('000000000000000010', 5805) ['00000']
('000000000000000011', 364) ['00000']
('000000000000000100', 9692) ['00000']
('000000000000000101', 247) ['00000']
('000000000000000110', 262) ['00000']
('000000000000000111', 5) ['00000']
('000000000000001000', 2010) ['00000']
('000000000000001001', 10) ['00000']
('000000000000001010', 22) ['00000']
('000000000000001011', 1) ['00000']
('000000000000001100', 46) ['00000']
('000000000000001101', 13) ['00000']
('000000000000001110', 13) ['00000']
('000000000000001111', 1) ['00000']
('000000000000010000', 2263) ['00000']
('000000000000010001', 4) ['00000']
('000000000000010010', 30) ['00000']
('000000000000010011', 1) ['00000']
('000000000000010100', 33) ['00000']
('000000000000010101', 2) ['00000']
('000000000000011000', 7) ['00000']
('000000000000011001', 1) ['00000']
('000000000000011101', 2) ['00000']
('000000000000011111', 11) ['00000']
('000000000000100000', 1856) [

In [33]:
def weight(string):
    return np.sum([int(x) for x in string])

logicalweighttarget = 1
keeplist = []
negateToLowestWeight = True
for row in sample:
    if weight(row[0][-8:]) == logicalweighttarget or weight(row[0][-8:]) == 8 - logicalweighttarget:
        det, readout = detector_builder(row[0], 5, 8, 1, mode = 'string')
        if negateToLowestWeight:
            if weight(negate(readout)) < weight(readout):
                keeplist.append((det, negate(readout), row[1]))
        else:
            keeplist.append((det, readout, row[1]))
keeplist = sorted(keeplist, key = lambda x: x[2], reverse = True)
for row in keeplist:
    print(row)


(['00000'], '00010000', 26397)
(['00000'], '00000001', 26323)
(['00000'], '00000001', 26154)
(['00000'], '00010000', 26020)
(['00000'], '00100000', 19836)
(['00000'], '00100000', 19642)
(['01100'], '00000100', 18704)
(['01100'], '00000100', 18440)
(['10000'], '00010000', 18238)
(['01000'], '00000001', 17968)
(['10000'], '00010000', 17933)
(['01000'], '00000001', 17834)
(['01001'], '00000001', 17434)
(['10001'], '00010000', 17406)
(['10001'], '00010000', 17324)
(['01001'], '00000001', 17087)
(['10100'], '01000000', 16416)
(['10100'], '01000000', 16067)
(['10010'], '00100000', 13797)
(['10010'], '00100000', 13754)
(['00010'], '00000010', 13547)
(['00010'], '00000010', 13410)
(['10011'], '00100000', 13385)
(['10011'], '00100000', 13369)
(['01010'], '00001000', 13275)
(['00000'], '01000000', 13141)
(['00000'], '01000000', 13074)
(['01010'], '00001000', 13004)
(['01010'], '00000010', 12906)
(['01010'], '00000010', 12630)
(['10110'], '10000000', 12405)
(['10110'], '10000000', 12324)
(['01001